## Mosaicing images from a STAC-API in GeoServer

Based on (https://docs.geoserver.org/2.25.x/en/user/community/stac-datastore/data-store.html#mosaicking-images-from-a-stac-store) \
Because work is already being done towards a STAC-API, it may be useful to use it to mosaic images instead of indexing them in GeoServer

## Using a STAC-API

Make sure you have the following installed: *pystac*

In [15]:
from pystac_client import Client


client = Client.open("https://earth-search.aws.element84.com/v1")


In [18]:
search = client.search(
    max_items=10,
    collections=['sentinel-2-pre-c1-l2a'],
)
print(f"{search.matched()} items found")


35015 items found


In [19]:
for item in search.items():
    print(item.id)

S2B_T21NYC_20221205T140704_L2A
S2B_T21NZC_20221205T140704_L2A
S2B_T22NBH_20221205T140704_L2A
S2B_T21NYD_20221205T140704_L2A
S2B_T21NZD_20221205T140704_L2A
S2B_T22NBJ_20221205T140704_L2A
S2B_T21NYE_20221205T140704_L2A
S2B_T21NZE_20221205T140704_L2A
S2B_T22NBK_20221205T140704_L2A
S2B_T21NYF_20221205T140704_L2A


## STAC in GeoServer

In [49]:
import requests

stac_url = "https://geoservice.dlr.de/eoc/ogc/stac/v1/"
response = requests.get(stac_url)
data = response.json()

# Print the first item to inspect its structure
print(data['features'][0])


KeyError: 'features'

In [47]:
import os
import zipfile
import requests

# GeoServer credentials
geoserver_url = "http://localhost:8080/geoserver"
geoserver_user = "admin"
geoserver_password = "geoserver"

# STAC datastore name
stac_store = "element84-stac"

# STAC Collection name
stac_collection = "sentinel-2-pre-c1-l2a"

# Mosaic Layer Name
mosaic_lyr = 'stac_mosaic'

# Define properties for indexer.properties
indexer_properties = f"""
MosaicCRS=EPSG:4326
TimeAttribute=datetime
AbsolutePath=true
Name={stac_collection}
Cog=true
Heterogeneous=true
HeterogeneousCRS=false
TypeName={stac_collection}
UseExistingSchema=true
LocationAttribute=assets/red/href
MaxInitTiles=10
"""
#! Error with datetime values : \"sentinel-1-grd\",\"node\":\"mmjhLtm2SICEHan_m0gQqQ\",\"reason\":{\"type\":\"parse_exception\",\"reason\":\"cannot parse empty date\"
#! might be because LocationAttribute is incorrect
# Define properties for datastore.properties
datastore_properties = f"""
StoreName=stac:{stac_store}
"""

# Create temporary directory for storing the properties files
temp_dir = "geoserver_temp"
os.makedirs(temp_dir, exist_ok=True)

# Write the properties files
with open(os.path.join(temp_dir, "indexer.properties"), "w") as f:
    f.write(indexer_properties)

with open(os.path.join(temp_dir, "datastore.properties"), "w") as f:
    f.write(datastore_properties)

# Create a zip file containing both properties files
zip_filename = "mosaic_configuration.zip"
zip_filepath = os.path.join(temp_dir, zip_filename)
with zipfile.ZipFile(zip_filepath, 'w') as zipf:
    zipf.write(os.path.join(temp_dir, "indexer.properties"), "indexer.properties")
    zipf.write(os.path.join(temp_dir, "datastore.properties"), "datastore.properties")

In [48]:

# Upload the zip file to GeoServer via REST API
upload_url = f"{geoserver_url}/rest/workspaces/your_workspace/coveragestores/{mosaic_lyr}/file.imagemosaic"
headers = {"Content-Type": "application/zip"}

with open(zip_filepath, 'rb') as zip_file:
    response = requests.put(upload_url, headers=headers, data=zip_file, auth=(geoserver_user, geoserver_password))

# Check if the request was successful
if response.status_code == 201:
    print("Mosaic layer successfully created.")
else:
    print(f"Failed to create mosaic layer. Status code: {response.status_code}")
    print(f"Response: {response.text}")


Mosaic layer successfully created.


In [46]:
# Clean up temporary files
os.remove(os.path.join(temp_dir, "indexer.properties"))
os.remove(os.path.join(temp_dir, "datastore.properties"))
os.remove(zip_filepath)
os.rmdir(temp_dir)

OSError: [WinError 145] The directory is not empty: 'geoserver_temp'